esta é a minha aplicação prática da criação de um agente react, tomando como base o que foi feito no vídeo.

as sessões 1 e 3 são idênticas as da aula, pois é apenas instalação do groq e configuração das funções do agente.

In [1]:
%pip install groq

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import os
os.environ['GROQ_API_KEY'] = "api key"

a primeira alteração vem nesta sessão, onde é instanciado um objeto do tipo Groq. no código da aula, foi solicitado um completions pedindo para explicar a importância de modelos de linguagens rápido. neste caso, não achei necessário fazer requisições do tipo, então apenas instanciei o objeto, sem pedir um completions. 

In [3]:
import os

from groq import Groq

client = Groq(
    api_key=os.environ.get("GROQ_API_KEY"),
)

In [4]:
class Agent:
    def __init__(self, client, system):
        self.client = client
        self.system = system
        self.messages = []
        if self.system is not None: 
            self.messages.append({"role": "system", "content": self.system})
        
    def __call__(self, message=""):
        if message:
            self.messages.append({"role": "user", "content": message})
        result = self.execute()
        self.messages.append({"role": "assistant", "content": result})
        return result
    
    def execute(self):
        completion = client.chat.completions.create(
            messages=self.messages,
            model="llama-3.3-70b-versatile",
        )
        return completion.choices[0].message.content

o system_prompt possui a mesma estrutura, onde fala pro agente trabalhar em um ciclo de pensamento, ação, pause e observação. no entanto, para a construção deste agente, fiz uma breve simulação de uma bicicletaria, onde são vendidas peças para manutenção. existe uma sessão para exemplo, seguindo o mesmo modelo da aula, mas considerando possíveis produtos da loja proposta. reiterando, o system_prompt é a primeira mensagem dada ao agente, para que ele saiba a forma como deve atuar, onde a sessão exemplo mostra uma simulação prática de uma consulta e a forma como o agente deveria trabalhar para atender a requisição.

In [5]:
system_prompt = """
Você opera em um ciclo de Pensamento, Ação, PAUSA, Observação.
Ao final do loop, você imprime uma resposta.
Use "Pensamento" para descrever seus pensamentos sobre a pergunta que lhe foi feita.

Use a opção Ação para executar uma das ações disponíveis e, em seguinda, retorne.
PAUSA

A observação será o resultado da execução desses ações.

As ações disponíveis são:

calcular:
Exemplo de cálculo: 4 * 7 / 3. Execute o cálculo e retorne o número - utiliza Python, portanto, certifique-se de usar a sintaxe de ponto flutuante, se necessário.

get_price: 
ex: get_price: Eixo
retorna o preço do Eixo em real.

Sessão de exemplo:

Pergunta: Qual é o preço de 2 Eixo e 2 Centrão?
Pensamento: Preciso descobrir o preço de 1 Eixo.
Ação: get_price: Eixo
PAUSA

Você receberá outra ligação com a seguinte mensagem:

Observação: A peça custa 50 reais
Pensamento: São 2 Eixo, então, preciso multiplicar o valor por 2.
Ação: calcular: 50 * 2
PAUSA

Você receberá outra ligação com a seguinte mensagem:

Observação: 2 Eixo custa 100 reais
Pensamento: Sei o preço de 2 Eixo, agora, preciso saber o preço de 1 Centrão
Ação: get_price: Centrão
PAUSA

Você receberá outra ligação com a seguinte mensagem:

Observação: A peça custa 30 reais
Pensamento: São 2 Centrão, então, preciso multiplicar o valor por 2.
Ação: calcular: 30 * 2
PAUSA

Você receberá outra ligação com a seguinte mensagem:

Observação: 2 Centrão custam 60 reais
Pensamento: Sei o preço de tudo que foi pedido, agora, preciso somar todos os valores.
Ação: calcular: 100 + 60
PAUSA

Se você tiver a resposta, mostre-a como "Resposta".

Resposta: O preço de 2 Eixo e 2 Centrão são 160 reais.

Agora é a sua vez.
""".strip()

as ferramentas funcionam de forma parecida. a ferramenta "calcular" funciona de forma idêntica ao "calculate" usado na aula, enquanto o "get_price" é um swich case, assim como o "get_planet_mass", mas as cases possuem valores diferentes, adaptando-se aos do contexto escolhido. a função calcular serve para calcular operações aritméticas, e o get_price para devolver o valor da peça consultada.

In [ ]:
#tools
def calcular(operation):
    return eval(operation)

def get_price(part) -> float:
    match part.lower():
        case "eixo":
            return 50
        case "centrão":
            return 30
        case "coroa":
            return 20
        case "direção":
            return 35
        case "corrente":
            return 15
        case "catraca":
            return 35
        case "cubo":
            return 100
        case _:
            raise 0.0
        

a função "repeticoes" funciona da mesma forma que o "agent_loop" da aula, mas usando as ferramentas definidas para este contexto. como parâmetros da função, temos a quantidade máxima de repetições que o loop deve fazer, o system_prompt e a query, que é a consulta feita pelo usuário. é instanciado um consultor e definidas as ferramantas que devem ser utilizadas. no loop, a variável result armazena o que foi devolvido pelo llama através do completions. caso tenha as palavras "PAUSA" e "Ação" no result, então uma ação deve ser executada, consultando a ferramenta necessárias. então a ferramenta é invocada e o resultado da requisição feita a ferramenta é armazenada, sendo a observação dada ao agente. se no result adquirido no início do loop existir a palavra "Resposta", então significa que a resposta já foi dada e o loop deve encerrar, por isso um break caso isso aconteça.

In [ ]:
import re

def repeticoes(max_iterations, system, query):
    consultor_vendas = Agent(client, system)
    tools = ['calcular', 'get_price']
    next_prompt = query
    i = 0
    
    while i < max_iterations:
        i += 1
        result = consultor_vendas(next_prompt)
        print(result)

        if"PAUSA" in result and "Ação" in result:
            action = re.findall(r"Ação: ([a-z_]+): (.+)", result, re.IGNORECASE)
            chosen_tool = action[0][0]
            argument = action[0][1]

            if chosen_tool in tools:
                result_tool = eval(f"{chosen_tool}('{argument}')")
                next_prompt = f"Observação: {result_tool}"
            else:
                next_prompt = "Observação: Ferramenta não encontrada"

            print(next_prompt)
            continue
    
        if "Resposta" in result:
            break

apenas um exemplo de execução.

In [8]:
repeticoes(max_iterations=15, system= system_prompt, query = "Qual é o preço 2 Coroa, 1 Eixo e 5 Corrente?")

Pensamento: Preciso descobrir os preços de cada peça e, em seguida, calcular o preço total. Vou começar com o preço de 1 Coroa, pois preciso saber o preço dela para poder calcular o preço de 2 Coroa.

Ação: get_price: Coroa
PAUSA
Observação: 20
Pensamento: Sei o preço de 1 Coroa, agora preciso calcular o preço de 2 Coroa. Para isso, vou multiplicar o preço de 1 Coroa por 2.

Ação: calcular: 20 * 2
PAUSA
Observação: 40
Pensamento: Agora sei o preço de 2 Coroa, que é 40 reais. Em seguida, preciso descobrir o preço de 1 Eixo para poderSomar ao preço total.

Ação: get_price: Eixo
PAUSA
Observação: 50
Pensamento: Agora sei o preço de 1 Eixo, que é 50 reais. Posso adicionar esse valor ao preço total. Além disso, ainda preciso descobrir o preço de 1 Corrente para calcular o preço de 5 Corrente.

Ação: get_price: Corrente
PAUSA
Observação: 15
Pensamento: Sei o preço de 1 Corrente, agora preciso calcular o preço de 5 Corrente. Para isso, vou multiplicar o preço de 1 Corrente por 5.

Ação: calcu

apenas um exemplo de execução.

In [9]:
repeticoes(max_iterations=15, system= system_prompt, query = "Qual é o preço 3 Catraca, 2 Direção, 1 Cubo e 10 Centrão?")

Pensamento: Preciso descobrir o preço de cada uma das peças solicitadas e, em seguida, calcular o preço total. Vou começar pelo Catraca, pois precisa saber o preço de 1 Catraca primeiro.

Ação: get_price: Catraca
PAUSA
Observação: 35
Pensamento: Agora que sei o preço de 1 Catraca, posso calcular o preço de 3 Catracas. Isso envolve multiplicar o preço de 1 Catraca por 3.

Ação: calcular: 35 * 3
PAUSA
Observação: 105
Pensamento: Sei o preço de 3 Catracas, agora preciso saber o preço de 1 Direção para, em seguida, calcular o preço de 2 Direções.

Ação: get_price: Direção
PAUSA
Observação: 35
Pensamento: Com o preço de 1 Direção, agora posso calcular o preço de 2 Direções. Isso é simplesmente multiplicar o preço de 1 Direção por 2.

Ação: calcular: 25 * 2 
PAUSA
Observação: 50
Pensamento: Agora sei o preço de 2 Direções, mas notei que havia um erro na multiplicação anterior, o preço de 1 Direção era 35, não 25. Vou corrigir isso calculando novamente o preço de 2 Direções com o valor corret